### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

    -Tracking agent behavior with logging, analytics, and debugging.
    -Transforming prompts, tool selection, and output formatting.
    -Adding retries, fallbacks, and early termination logic.
    -Applying rate limits, guardrails, and PII detection.


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

1. Long-running conversations that exceed context windows.
2. Multi-turn dialogues with extensive history.
3. Applications where preserving full conversation context matters.

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message based summarization

agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger = ("messages", 10),
            keep=("messages",4)
        )
    ]
)

In [7]:
# Run with the thread id

config={"configurable":{"thread_id":"test-1"}}

In [12]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response["messages"])}")

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to receive answers to basic arithmetic operations.\n\n## SUMMARY\nThe conversation history includes a series of simple math problems with their respective solutions: 2+2=4, 10*5=50, 100/4=25, 15-7=8, and 3*3. The user has repeated some math problems, with the answers consistently being provided.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe user has asked multiple math problems, and the next step is to await a new math problem from the user or assist with a different math question if needed.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='e8ad4436-e645-4414-941d-c4468f534808'), AIMessage(content="3 * 3 = 9. Simple multiplication. What's the next math problem you'd like to solve?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 300, 'total_tokens': 324, 'completion_tim

### Token Size

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger = ("messages", 100),
            keep=("messages",50)
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [16]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~203 tokens, 8 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='a6904886-80f1-4d23-abc8-a53978e0b082'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ayb2e0j36', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 226, 'total_tokens': 241, 'completion_time': 0.027895819, 'completion_tokens_details': None, 'prompt_time': 0.011576564, 'prompt_tokens_details': None, 'queue_time': 0.057267036, 'total_time': 0.039472383}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eebad-40e5-7d11-91b4-eff670977b35-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'ayb2e0j36', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

### Fraction

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # groq:llama-3.3-70b-versatile context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~81 tokens (0.0633%), 8 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='194e80cd-673d-4bee-aedc-3e7e3bbd5bbd'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '7qartjqxq', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 210, 'total_tokens': 225, 'completion_time': 0.041190985, 'completion_tokens_details': None, 'prompt_time': 0.010288507, 'prompt_tokens_details': None, 'queue_time': 0.056359263, 'total_time': 0.051479492}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eebb2-7f94-71d0-b61e-40ddf14a1897-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '7qartjqxq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada